# Causeway duration forecasting (canonical `travel_times`)

This notebook loads the **canonical** BigQuery table `swiftborder.causeway.travel_times` (live download with local CSV cache), then runs:

1. **XGBoost** (sklearn) — 60-minute horizon, `jb_to_woodlands` / `SG_TO_MY`
2. **LSTM** (optional) — same tensors via [`timeseries_lstm.py`](../timeseries_lstm.py); plug in a **custom recurrent layer** when ready

Production serve remains **30-minute BQML** (`v_forecast_recent`). Score that with [`layer_b.py`](../layer_b.py), not this notebook.


## 1. Canonical dataset (BigQuery + cache)

Set `REFRESH_FROM_BQ = True` to re-pull the full table. Otherwise the notebook reuses `../data/causeway_gdata.csv`.


In [ ]:
from pathlib import Path
import sys

sys.path.insert(0, str(Path.cwd().parent))

import timeseries_xgb as tsx
from timeseries_xgb import TimeSeriesConfig

PROJECT = "swiftborder"
CONFIG = TimeSeriesConfig()
CACHE_PATH = Path("../data/causeway_gdata.csv")
REFRESH_FROM_BQ = False  # True = live download of full travel_times

export = tsx.sync_canonical_travel_times(CACHE_PATH, project=PROJECT, refresh=REFRESH_FROM_BQ)
raw = tsx.prepare_route_frame(export, CONFIG)
print("canonical rows:", len(export), "| route rows:", len(raw), "| route:", CONFIG.route_id)
print("cache:", CACHE_PATH.resolve())


## 2. XGBoost (60-minute horizon)


In [ ]:
features = tsx.engineer_features(raw, CONFIG)
X_train, X_test, y_train, y_test = tsx.build_supervised_matrices(features, CONFIG)
xgb_model = tsx.train_xgb(X_train, y_train)
pred = xgb_model.predict(X_test)
print("XGB hold-out RMSE (min):", round(tsx.rmse_minutes(y_test, pred), 2))


In [ ]:
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import pandas as pd

y_series, free_flow = tsx.regularized_series(raw, CONFIG)
dates = ["2026-09-22", "2026-09-23", "2026-09-24"]
scores = tsx.score_forecast_days(xgb_model, y_series, free_flow, dates, CONFIG)
print("RMSE (minutes)")
print(scores.pivot(index="method", columns="date", values="RMSE_min").round(2))

fig, axes = plt.subplots(1, 3, figsize=(24, 6), sharey=True)
for ax, d in zip(axes, dates):
    day_idx = pd.date_range(pd.Timestamp(d), periods=288, freq="5min")
    actual = y_series.reindex(day_idx) / 60
    ax.plot(actual.index, actual, color="black", lw=1.8, label="Actual")
    ax.set_title(d)
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))
    ax.grid(alpha=0.3)
axes[0].set_ylabel("Minutes")
fig.suptitle(f"JB to Woodlands — {CONFIG.horizon_minutes} min ahead (XGB)")
plt.tight_layout()
plt.show()


## 3. LSTM (custom layer hook)

Requires `pip install tensorflow` (see `requirements-notebook.txt`). Uses the **same** `features` frame and chronological split as XGB.

Replace `ResidualGatedLSTM` with your custom `keras.layers.Layer` design (e.g. multi-scale gates, attention over lags).


In [ ]:
RUN_LSTM = False  # set True after installing tensorflow

if RUN_LSTM:
    import timeseries_lstm as tsl
    from tensorflow.keras import layers

    class ResidualGatedLSTM(layers.Layer):
        """Example custom block: LSTM + layer norm (swap for your architecture)."""

        def __init__(self, units: int, **kwargs):
            super().__init__(**kwargs)
            self.units = units
            self.lstm = layers.LSTM(units, return_sequences=False)
            self.norm = layers.LayerNormalization()

        def call(self, inputs, training=None):
            return self.norm(self.lstm(inputs, training=training))

    seq_cols = tsl.sequence_feature_columns(features, CONFIG)
    X_tr, X_te, y_tr, y_te = tsl.build_lstm_sequences(features, CONFIG, sequence_cols=seq_cols)
    lstm_model = tsl.build_lstm_model(
        input_shape=(X_tr.shape[1], X_tr.shape[2]),
        lstm_units=64,
        custom_recurrent_layer=ResidualGatedLSTM,
    )
    lstm_model.fit(X_tr, y_tr, validation_data=(X_te, y_te), epochs=20, batch_size=64, verbose=1)
    lstm_pred = lstm_model.predict(X_te, verbose=0)
    print("LSTM metrics (min):", tsl.evaluate_predictions(y_te, lstm_pred))
else:
    print("LSTM skipped. Set RUN_LSTM=True and install tensorflow to train.")
